<a href="https://colab.research.google.com/github/SY-256/anomaly_detection/blob/main/notebook/chapter7_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 統計モデリングを用いた手法

In [ ]:
# サンプルデータの作成用関数
import numpy as np
import pandas as pd
from scipy import stats

### データ生成関数
def generate_usedcar_data(
        intercept, b_models, b_year, b_odometer, sigma_p,
        sigma_g, mu_distances, s_distance, sample_size, random_state
):
    ### 説明変数の生成
    # 車種（各車種1/3ずつ）
    size_model = int(sample_size / 3)
    x_model = np.concatenate([np.full(size_model, 0),
                              np.full(size_model, 1),
                              np.full(sample_size - size_model*2, 2)])
    # ordinal encoding前の車種
    model_list = ["sedan", "compact", "suv"]
    model_names = np.vectorize(lambda x: model_list[x])(x_model)
    # 経過年数（1~10年の整数を一様分布から生成）
    x_year = stats.randint.rvs(low=1, high=11, size=sample_size, random_state=random_state)
    # 年間走行距離（対数正規分布から生成）
    scales = np.vectorize(lambda x: np.exp(mu_distances[x]))(x_model)
    distance_per_year = stats.lognorm.rvs(
        s=s_distance, scale=scales, size=sample_size, random_state=random_state
    )
    # 走行距離（経過年数x 年間走行距離）
    x_odometer = x_year * distance_per_year
    # 販売店（インデックスを25で割った余りから計算）
    dealers = np.array([i % 25 for i in range(sample_size)])
    ### 応答変数の生成
    r_g = stats.norm.rvs(loc=0, scale=sigma_g, size=25, random_state=random_state)
    # 価格の平均
    mu_prices = intercept \
                + np.vectorize(lambda x: b_models[x])(x_model) \
                + x_year * b_year \
                + x_odometer * b_odometer \
                + np.vectorize(lambda x: r_g[x])(dealers)
    # 正規分布から価格を生成（対数正規分布と乱数シードを変える）
    prices = mu_prices + stats.norm.rvs(
        loc=0, scale=sigma_p, size=sample_size, random_state=random_state+2
    )
    # pandas.DataFrameにまとめる
    df = pd.DataFrame({"model_name": model_names,
                       "year": x_year,
                       "dealer": np.vectorize(lambda x: f'dealer{x}')(dealers),
                       "price": prices})

    return df

- 関数使ってサンプルデータを生成

In [ ]:
from sklearn.model_selection import train_test_split

### 正常モデルのパラメータ
intercept_norm = 400 # 切片 w_0 [万円]
b_models_norm = [0, -50.0, 100.0] # 各車種の係数 w_d[i] [万円]
b_year_norm = -10.0 # 経過年数の比例係数 w_1 [万円/年]
b_odometer_norm = -5.0 # 走行距離の比例係数 w_2 [万円/万km]
sigma_price_norm = 20 # 個々の車のばらつき（標準偏差） σ_i [万円]
sigma_dealer_norm = 15 # 販売店ごとのばらつき（標準偏差） σ_g [万円]
# 年間走行距離生成用の対数正規分布のパラメータ
mu_distances_norm = [-0.5, -1.0, 0] # 対数正規分布のパラメータμ（車種ごとに異なる）
s_distance_norm = 0.2 # 対数正規分布のパラメータσ（exp(μ+σ^2/2)が期待値）

### データの生成
# 正常データ生成
df_normal = generate_usedcar_data(
    intercept_norm, b_models_norm, b_year_norm, b_odometer_norm, sigma_price_norm,
    sigma_dealer_norm, mu_distances_norm, s_distance_norm, sample_size=1500, random_state=42
)
# 異常データ生成（正常データよりも100万円高い平均値）
df_anomaly_upper = generate_usedcar_data(
    intercept_norm + 100, b_models_norm, b_year_norm, b_odometer_norm, sigma_price_norm, sigma_dealer_norm,
    mu_distances_norm, s_distance_norm, sample_size=75, random_state=42
)
# 異常データ生成（正常データよりも100万円低い平均値）
df_anomaly_lower = generate_usedcar_data(
    intercept_norm - 100, b_models_norm, b_year_norm, b_odometer_norm, sigma_price_norm, sigma_dealer_norm,
    mu_distances_norm, s_distance_norm, sample_size=75, random_state=42
)
df_anomaly = pd.concat([df_anomaly_upper, df_anomaly_lower], axis=0)
# 正常データと異常データにラベルを付ける
df_normal["label"] = "normal"
df_anomaly["label"] = "anomaly"
# 学習データと推論データを分ける（2/3が学習用、1/3が推論用）
df_norm_train, df_norm_inference = train_test_split(df_normal, train_size=2/3, random_state=42)
df_anom_train, df_anom_inference = train_test_split(df_anomaly, train_size=2/3, random_state=42)
# 正常データと異常データを合体させる
df_train = pd.concat([df_norm_train, df_anom_train], axis=0)
df_train = df_train.reset_index(drop=True)
df_inference = pd.concat([df_norm_inference, df_anom_inference], axis=0)
df_inference = df_inference.reset_index(drop=True)
# 作成したデータをCSVで保存
df_train.to_csv("./usedcar_dataset_train.csv", index=False)
df_inference.to_csv("./usedcar_dataset_inference", index=False)
# 作成した学習用データセットの情報を表示
df_train.info()

## 乱数シードに関する注意
- `stats.norm.rvs`や`stats.lognorm.rvs`で乱数を生成する際に、複数の分布に同じ乱数を与えると、意図せずに相関が生じることがある
- 相関を生じさせたくない場合は異なる乱数シードを設定する必要がある

In [ ]:
# 同じ乱数シードを使用した場合の確率変数同士の相関の発生
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_lognorm_norm_rvs(lognorm_seed, norm_seed):
    lognorm_rvs = stats.lognorm.rvs(
        s=0.3, scale=np.exp(0), size=1000, random_state=lognorm_seed
    )
    norm_rvs = stats.norm.rvs(
        loc=0, scale=20, size=1000, random_state=norm_seed
    )
    sns.scatterplot(x=lognorm_rvs, y=norm_rvs, c="#555555")
    plt.show()

plot_lognorm_norm_rvs(42, 42) # 同じ乱数シード
plot_lognorm_norm_rvs(42, 43) # 異なる乱数シード

- 同じ乱数シードを使用した場合は対数正規分布と正規分布から生成した確率変数に相関がある
- 異なる乱数シードを使用すると無相関の独立な値を生成できる
- 意図しない相関を防ぐために、階層モデルにように複数の確率分布を組み合わせる場合は、分布ごとに異なる乱数シードを設定することが望ましい

## 入出力のあるデータの可視化
- 可視化やEDAを用いて、モデル構造の当たりを付ける
- pairplotを使ってデータを可視化する

In [ ]:
# pairplot による入出力があるデータの可視化
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# CSVからPandas DataFrameにデータ読み込み
df = pd.read_csv('https://raw.githubusercontent.com/ghmagazine/python_anomaly_detection_book/refs/heads/main/notebooks/datasets/usedcar_dataset_train.csv')
df_normal = df[df["label"] == "normal"]
# pairplotによる可視化
sns.pairplot(
    data=df_normal,
    hue="model_name",
    palette=["#aaaaaa", "#666666", "#111111"]
)
plt.show()

- 最下段の散布図から、応答変数である中古車価格（`price`）と他の説明変数（数値変数`year`、`odometer`）との関係を見ると、概ね線形の関係を確認できる
- 線形関係が確認できるため、一般線形モデル（GLM）のようなシンプルなモデルで挙動をある程度正確に表現できる見通しが立つ
- 単回帰直線と箱ひげ図を用いた可視化の方法

In [ ]:
# 箱ひげ図ち単回帰直線による入出力があるデータの可視化
# 描画用のFigureとAxesを生成
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))
# regplotによる単回帰直線と散布図の描画
for i, colname in enumerate(["year", "odometer"]):
    sns.regplot(
        data=df_normal,
        x=colname,
        y="price",
        scatter_kws={"color": "#999999"}, # 点の色
        line_kws={"color": "#111111"}, # 線の色
        ci=None, # 回帰直線の信頼区間の表示有無
        ax=axes[i] # 描画対象のAxes
    )

# boxplotによる箱ひげ図に描画
sns.boxplot(
    data=df_normal,
    x="model_name",
    y="price",
    color="#999999",
    ax=axes[2]
)
plt.show()

- 単回帰直線は説明変数（数値）による応答変数の変化の把握に、箱ひげ図は説明変数（カテゴリ）による応答変数の分布の差の把握に、それぞれ効果的な手法

## 7.3 1変数線形回帰モデルによる異常検知
- 最尤推定による学習を用いて、異常検知を実装
- 最尤推定による線形回帰モデルのパラメータ推定、および分位点に基づく異常度の閾値を算出
- statsmodelsの`statsmodels.regression.linear_model.OLS`クラスを用いると、線形回帰モデルを簡単に実装できる

In [ ]:
# 線形回帰による1 説明変数の異常検知の実装例（学習）
import pandas as pd
import numpy as np
import statsmodels.api as sm

### 学習データの読み込みと前処理
df = pd.read_csv('https://raw.githubusercontent.com/ghmagazine/python_anomaly_detection_book/refs/heads/main/notebooks/datasets/usedcar_dataset_train.csv')
# 正常データのみ抽出
df_normal = df[df["label"] == "normal"]
# 学習データの説明変数（'year'）と応答変数（'price'）を別々に保持（応答変数のみndarray化）
x_train = df_normal["year"]
y_train = df_normal["price"].to_numpy()

### 学習ステップ1. 正常のモデルを作成する
X_train_intercept = sm.add_constant(x_train) # 切片を追加
mod = sm.OLS(y_train, X_train_intercept) # 線形回帰モデル（OLS）を作成
res = mod.fit() # モデルの学習を実行
w_1 = res.params[1] # パラメータw_1
w_0 = res.params[0] # パラメータw_0
sigma2 = res.scale*res.df_resid / mod.nobs # パラメータσ^2（scaleは不偏推定値）

### 学習ステップ2. 異常度を定義する
# 式を定義するのみで処理は実施しない

### 学習ステップ3. 異常度に閾値を設ける
TARGET_FP_RATE = 0.0027 # ターゲットとする誤報率（正規分布の3σ相当=0.0027）
x_train_anom_score = (y_train-w_1*x_train-w_0)**2 / sigma2 # 異常度を求める
# 異常度の分位点から閾値算出
a_th = np.quantile(x_train_anom_score, 1-TARGET_FP_RATE)

# 学習で求めたパラメータをすべて表示
print(f"w_1={w_1}")
print(f"w_0={w_0}")
print(f"sigma2={sigma2}")
print(f"a_th={a_th}")

### statsmodelsで線形回帰を実装する際の注意点
- 切片$w_0$をモデルに含めるためには、事前に`sm.add_constant`関数で説明変数に定数項を追加する必要がある
- 学習で求められたパラメータは、学習結果（`res`インスタンス）のメンバ変数`params`に格納される
- 残差の分散`scale`は、分母$N$（サンプルサイズ）の代わりに$N-M-1$（$M$は説明変数の数）で割られた不偏推定値となる。よってパラメータ$\hat{\sigma}^2$を求めるためには、`scale`を$\frac{N-M-1}{N}$倍する必要がある
- $N-M-1$は`res.df_resid`変数から取得できる
- statsmodelsでは、学習結果のインスタンス`res`から`summary`メソッドを実行することで、モデルの概要を表示できる

In [ ]:
# summary()メソッドによるモデル概要の表示
print(res.summary())

- 推定されたパラメータから求めた確率密度関数$p(y \mid x, \hat{w}_1, \hat{w}_0, \hat{\sigma}^2)$と学習データを重ねてプロット

In [ ]:
# 学習した線形回帰モデルの確率密度関数と学習データを重ねてプロット
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats

### 確率密度関数を描画
# (x,y)格子点を作成
x_min, x_max = np.min(x_train), np.max(x_train)
y_min, y_max = np.min(y_train), np.max(y_train)
x_grid = np.arange(x_min - 1, x_max + 2)
y_grid = np.linspace(y_min - 50, y_max + 50, num=200)
X, Y = np.meshgrid(x_grid, y_grid)
XY_grid = np.c_[X.ravel(), Y.ravel()]
# 確率密度関数
mu_grid = w_1 * XY_grid[:, 0] + w_0 # 線形予測子で平均μを予測
X_grid_pd = stats.norm.pdf(XY_grid[:, 1], loc=mu_grid, scale=np.sqrt(sigma2))
# 確率密度をプロット
X_grid_pivot = X_grid_pd.reshape(X.shape) # ピボット化
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))
ax.contourf(X, Y, X_grid_pivot, levels=10, cmap=cm.gray, alpha=0.5)

### 学習データを散布図で描画
sns.scatterplot(x=x_train, y=y_train, c="#333333", ax=ax, s=18, marker="o")
ax.set_xlabel("year")
ax.set_ylabel("price")
plt.show()

- 横軸に説明変数`year`、縦軸に応答変数`price`をとっている
- データにフィットした確率密度関数が推定できていることがわかる

## 推論
- 推論データに対する異常度の算出と異常判定を行う

In [ ]:
# 線形回帰による1 説明変数の異常検知の実装例
### 学習したパラメータを記載
W_1=-12.778952686336028 # 線形予測子の係数パラメータ
W_0=409.52848846804113 # 線形予測子の切片パラメータ
SIGMA2=3583.7515391302095 # 残差の分散パラメータσ^2
A_TH=4.916560173082554 # 異常度のしきい値

### 推論データの読み込みと前処理
# CSVからPandas DataFrameにデータ読み込み
df_inference = pd.read_csv('https://raw.githubusercontent.com/ghmagazine/python_anomaly_detection_book/refs/heads/main/notebooks/datasets/usedcar_dataset_inference.csv')
# 推論データの説明変数（'year'）と応答変数（'price'）をNumpyのndarray化
x_inference = df_inference["year"].to_numpy()
y_inference = df_inference["price"].to_numpy()

### 推論を実行
# 異常度を算出
anomaly_scores = (y_inference-W_1*x_inference-W_0)**2 / SIGMA2
# 閾値判定により異常の有無を判定
pred = np.where(anomaly_scores > A_TH, "anomaly", "normal")
# 推論結果を表示
print(pred)

- 推論結果の決定境界（異常と正常の判定の境界）を可視化

In [ ]:
# 推論結果の決定境界を可視化
# 描画用のFigureとAxesを生成
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5, 5))

### 正常と異常の範囲を色分け
# (x,y)格子点を作成
x_min, x_max = np.min(x_inference), np.max(x_inference)
y_min, y_max = np.min(y_inference), np.max(y_inference)
x_grid = np.arange(x_min-1, x_max+2)
y_grid = np.linspace(y_min-50, y_max+50, num=500)
X, Y = np.meshgrid(x_grid, y_grid)
XY_grid = np.c_[X.ravel(), Y.ravel()]
# 異常度を算出
anomaly_scores_grid = (XY_grid[:, 1]-W_1*XY_grid[:, 0]-W_0)**2 / SIGMA2
# 閾値判定
pred_grid = np.where(anomaly_scores_grid > A_TH, 0, 1)
# 正常と異常の境界をプロット
pred_pivot = pred_grid.reshape(X.shape)
ax.contourf(X, Y, pred_pivot, levels=1,
            cmap=cm.gray, alpha=0.5)

### 各データを散布図としてプロット
sns.scatterplot(data=df_inference, x="year", y="price",
                hue="label", palette=["#999999", "#111111"], ax=ax)
# 凡例を追加
ax.legend()
plt.show()

- 正常範囲の中央に位置する異常データを多数検知できていないことがわかる（見逃している）
- 車種（`model_name`）ごとに正常範囲が異なるにもかかわらず、車種を区別せずに一つのモデルで扱ったため
- 車種ごとに正常範囲の違いをモデルに反映する必要がある

## 7.4 多変数線形回帰モデルによる異常検知
- 一般的に複数の説明変数を加えた方が応答変数をより正確にモデル化できる
- 最尤推定によりモデルパラメータを求め、異常度を設定して判定を行う（1変数線形回帰と同じ）

### 箱ひげ図によるカテゴリ変数の寄与確認
- 箱ひげ図による可視化を通して、カテゴリ変数`model_name`（車種）が応答変数`price`（中古車価格）に変化に寄与するか確認

In [ ]:
# 箱ひげ図によるカテゴリ変数の寄与の確認

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# CSVからPandas DataFrameにデータ読み込み
df = pd.read_csv('https://raw.githubusercontent.com/ghmagazine/python_anomaly_detection_book/refs/heads/main/notebooks/datasets/usedcar_dataset_train.csv')
# 正常データのみ使用
df_normal = df[df["label"] == "normal"]

# boxplotによる箱ひげ図の描画
sns.boxplot(
    data=df_normal,
    x="model_name",
    y="price",
    color="#999999"
)
plt.show()

- カテゴリ変数`model_name`によって、明確に応答変数が変化しているように見えるため、`model_name`は説明変数に加えた方が良い
- 最終判断は他の説明変数との関係性も可視化した上で、総合的に判断する

### 散布図と単回帰直線による目的変数との相関の確認
- 散布図と単回帰直線による可視化で、各説明変数候補と目的変数`price`との相関関係を確認し、相関があるものを説明変数として採用する

In [ ]:
# 散布図と単回帰直線による目的変数との相関の確認
# 描画用のFigureとAxesを生成
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))
# regplotによる単回帰直線と散布図の描画
for i, colname in enumerate(["year", "odometer"]):
    sns.regplot(
        data=df_normal,
        x=colname,
        y="price",
        scatter_kws={"color": "#999999"}, # 点の色
        line_kws={"color": "#111111"}, # 線の色
        ci=None, # 回帰直線の信頼区間の表示の有無
        ax=axes[i]
    )

- `odometer`と`price`は一見すると相関が無いように見えるが、散布図の分布が複数のクラスタに分かれており、クラスタ内では左上から右下にかけて相関関係があるように見える

In [ ]:
# `model_name`（車種）ごとに分けて散布図と単回帰直線の描画
# model_nameごとにデータを分割してループ
for j, (model_name, data_model) in enumerate(df_normal.groupby('model_name')):
    # 描画用のFigureとAxesの生成
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
    # regplotによる単回帰直線と散布図の描画
    for i, colname in enumerate(["year", "odometer"]):
        sns.regplot(
            data=data_model,
            x=colname,
            y="price",
            scatter_kws={"color": "#999999"},
            line_kws={"color": "#111111"},
            ci=None,
            ax=axes[i]
        )
        # x軸、y軸範囲を統一
        axes[i].set_xlim(df_normal[colname].min(), df_normal[colname].max())
        axes[i].set_ylim(df_normal["price"].min(), df_normal["price"].max())
    plt.suptitle(f"model_name: {model_name}")
    plt.show()

- `model_name`（車種）ごとにプロットをわけることで、`odometer`と応答変数`price`の相関関係が明確になった
- `year`、`odometer`どちらも説明変数に加えた方が良い

### 散布図による正常と異常の分離性の確認
- 異常データが取得できる場合に限るが、散布図を用いて正常と異常の分離性も確認

In [ ]:
# 散布図による正常と異常の分離性の確認
# 描画用FigureとAxesを生成
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))
# regplotによる単回帰直線と散布図の描画
for i, colname in enumerate(["year", "odometer"]):
    sns.scatterplot(
        data=df,
        x=colname,
        y="price",
        hue="label",
        palette=["#999999", "#111111"],
        ax=axes[i]
    )

- `model_name`（車種）を区別しない場合、正常と異常を上手く分離できない

In [ ]:
# `model_name`（車種）ごとにわけて散布図による正常と異常の分離性を確認
# model_nameごとにデータを分割してループ
for j, (model_name, data_model) in enumerate(df.groupby("model_name")):
    # 描画用のFigureとAxesを生成
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
    # regplotによる単回帰直線と散布図の描画
    for i, colname in enumerate(["year", "odometer"]):
        sns.scatterplot(
            data=data_model,
            x=colname,
            y="price",
            hue="label",
            palette=["#999999", "#111111"],
            ax=axes[i]
        )
        # x軸、y軸範囲を統一
        axes[i].set_xlim(df[colname].min(), df[colname].max())
        axes[i].set_ylim(df["price"].min(), df["price"].max())
    plt.suptitle(f"model_name: {model_name}")
    plt.show()

- 正常と異常がうまく分離していそう
- `year`（経過年数）、`odometer`（総走行距離）、`model_name`（車種）の3変数すべてを説明変数として用いることで、ある程度の性能の異常検知の実現が期待できる

## モデルの学習
- 最尤推定による線形回帰モデルのパラメータ推定
- ある分位点に基づく異常度の閾値の算出

In [ ]:
# 線形回帰による多説明変数の異常検知の実装例
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import statsmodels.api as sm

### 学習データの読み込みと前処理
df = pd.read_csv('https://raw.githubusercontent.com/ghmagazine/python_anomaly_detection_book/refs/heads/main/notebooks/datasets/usedcar_dataset_train.csv')
# 正常データのみ抽出
df_normal = df[df["label"] == "normal"]
# カテゴリ変数'model_name'の数値化（one-hot encoding）
enc = OneHotEncoder(drop="first")
model_name_cat = enc.fit_transform(df_normal[["model_name"]].to_numpy()).toarray()
model_cat_names = [f"model_name_{col}" for col in enc.categories_[0][1:]]
df_model_name = pd.DataFrame(model_name_cat, columns=model_cat_names)
df_encoded = pd.concat([df_normal.drop("model_name", axis=1), df_model_name], axis=1)
# 学習データの説明変数と応答変数（`price`）を別々に保持
X_train = df_encoded[["year", "odometer"]+model_cat_names]
y_train = df_encoded["price"].to_numpy()

### 学習ステップ1. 正常のモデルを作成する
X_train_intercept = sm.add_constant(X_train) # 切片を追加
mod = sm.OLS(y_train, X_train_intercept) # 線形回帰モデル（OLS）を作成
res = mod.fit() # モデルの学習を実行
w = res.params[1:].to_numpy() # パラメータw
w_0 = res.params[0] # パラメータw_0
sigma2 = res.scale*res.df_resid / mod.nobs # パラメータσ^2（scaleは不偏推定量）

### 学習ステップ2. 異常を表す指標（異常度）を定義
# 式を定義するのみで処理は実施しない

### 学習ステップ3. 異常度に閾値を設ける
TARGET_FP_RATE = 0.0027 # ターゲットとする誤報率（正規分布の3σ相当）
# 異常度を求める
x_train_anom_score = (y_train-w@X_train.T.to_numpy()-w_0)**2 / sigma2
# 異常度の分位点から閾値算出
a_th = np.quantile(x_train_anom_score, 1-TARGET_FP_RATE)

### 学習でもとめたパラメータを表示
print(f"w={w}")
print(f"w_0={w_0}")
print(f"sigma2={sigma2}")
print(f"a_th={a_th}")